# 깊게 쌓으면 왜 더 나빠지는가 — degradation 재현

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/No1Joon/oh-my-notebooks/blob/main/notebooks/resnet_degradation.ipynb)

Deep Residual Learning for Image Recognition
([arXiv 1512.03385](https://arxiv.org/abs/1512.03385)) 의 핵심 관찰을
CIFAR-10 으로 직접 확인합니다.

같은 조건에서 네 모델을 끝까지 돌립니다.

| 모델 | 잔차 연결 | 깊이 |
|---|---|---|
| plain-20 | 없음 | 20 |
| plain-56 | 없음 | 56 |
| resnet-20 | 있음 | 20 |
| resnet-56 | 있음 | 56 |

보려는 것은 두 가지입니다.

1. **degradation** — 잔차 연결이 없으면 56층이 20층보다 나쁜가.
2. **역전** — 잔차 연결을 넣으면 56층이 20층보다 좋아지는가.

논문 Figure 를 옮겨 오지 않습니다. 아래 셀이 실제로 뽑은 값만 씁니다.

`런타임 → 런타임 유형 변경 → T4 GPU` 를 켜고 실행하세요.
CPU 로도 돌아가지만 몇 시간이 걸립니다.

In [ ]:
import math
import os
import time

import torch
import torchvision

print("torch       :", torch.__version__)
print("torchvision :", torchvision.__version__)

dev = "cuda" if torch.cuda.is_available() else "cpu"
amp = dev == "cuda"
print("device      :", dev)
if dev == "cuda":
    p = torch.cuda.get_device_properties(0)
    print("gpu         :", p.name)
    print("memory (MiB):", p.total_memory // 1024 ** 2)

## 데이터 — CIFAR-10

출처: Alex Krizhevsky, *Learning Multiple Layers of Features from
Tiny Images* (2009). 32x32 컬러 이미지 10개 범주, 학습 50,000장 ·
테스트 10,000장.

라이선스: 저작자가 연구·교육 목적 이용을 허용한 공개 데이터셋
입니다. 저장소에 담지 않고 실행 시점에 내려받습니다.

원본 배포처(`cs.toronto.edu`)는 Colab 에서 초당 70KB 남짓밖에
안 나와 170MB 받는 데 40분이 걸립니다. 같은 데이터를 그대로
옮겨 둔 Hugging Face 미러 `uoft-cs/cifar10` 에서 받습니다 —
10초면 끝납니다. 내용은 원본과 같습니다.

6만 장을 uint8 로 들고 있어도 176MiB 라 통째로 가속기에
올려 둡니다.

In [ ]:
import io
import urllib.request

import numpy as np
import pyarrow.parquet as pq
import torch.nn.functional as F
from PIL import Image

BASE = ("https://huggingface.co/datasets/uoft-cs/"
        "cifar10/resolve/main/plain_text/")
FILES = {"train": "train-00000-of-00001.parquet",
         "test": "test-00000-of-00001.parquet"}
CLASSES = ["airplane", "automobile", "bird", "cat",
           "deer", "dog", "frog", "horse", "ship",
           "truck"]


def fetch(split):
    t0 = time.time()
    raw = urllib.request.urlopen(BASE + FILES[split]).read()
    rows = pq.read_table(io.BytesIO(raw)).to_pylist()
    xs = np.stack([
        np.array(Image.open(io.BytesIO(r["img"]["bytes"]))
                 .convert("RGB"))
        for r in rows])
    ys = np.array([r["label"] for r in rows],
                  dtype=np.int64)
    print(f"{split:<6} {len(ys):>6}장"
          f"  {time.time() - t0:5.1f}s"
          f"  {xs.nbytes / 1024 ** 2:.0f} MiB")
    return (torch.from_numpy(xs).to(dev),
            torch.from_numpy(ys).to(dev))


train_x, train_y = fetch("train")
test_x, test_y = fetch("test")
print("classes :", len(CLASSES))

증강은 논문과 같습니다 — 4픽셀 패딩 후 32x32 무작위 크롭,
좌우 뒤집기. 다만 CPU 가 아니라 **GPU 에서 배치 단위로**
합니다. Colab 은 vCPU 가 2개뿐이라 CPU 로 증강하면 GPU 가
데이터를 기다리며 놉니다 — 같은 학습이 열 배 넘게 느려집니다.

In [ ]:
MEAN = (0.4914, 0.4822, 0.4465)
STD = (0.2470, 0.2435, 0.2616)
mean = torch.tensor(MEAN, device=dev).view(1, 3, 1, 1)
std = torch.tensor(STD, device=dev).view(1, 3, 1, 1)


def to_float(u8):
    return u8.permute(0, 3, 1, 2).float().div_(255)


def normalize(u8):
    return to_float(u8).sub_(mean).div_(std)


def augment(u8):
    x = F.pad(to_float(u8), (4, 4, 4, 4))
    n = x.shape[0]
    ar = torch.arange(32, device=dev)
    oy = torch.randint(0, 9, (n, 1), device=dev)
    ox = torch.randint(0, 9, (n, 1), device=dev)
    rows = (oy + ar)[:, :, None].expand(n, 32, 32)
    cols = (ox + ar)[:, None, :].expand(n, 32, 32)
    bi = torch.arange(n, device=dev)[:, None, None]
    x = x[bi, :, rows, cols].permute(0, 3, 1, 2)
    flip = torch.rand(n, 1, 1, 1, device=dev) < 0.5
    x = torch.where(flip, x.flip(-1), x)
    return x.sub_(mean).div_(std)


xb = augment(train_x[:256])
print("배치     :", tuple(xb.shape), xb.dtype)
print("평균/표준편차 : %.3f / %.3f"
      % (xb.mean().item(), xb.std().item()))

## 모델 — 블록 하나만 다르다

논문의 CIFAR 구성 그대로입니다. 3x3 합성곱만 쓰고, 채널은
16 → 32 → 64 로 늘리며 각 단계마다 해상도를 반으로 줄입니다.
깊이 `d` 는 각 단계에 `(d-2)/6` 개의 블록을 둡니다.

`PlainBlock` 과 `ResidualBlock` 은 상속 관계입니다. 아래에서
두 `forward` 의 차이를 그대로 찍어 봅니다.

In [ ]:
import torch.nn as nn


def conv3(cin, cout, stride=1):
    return nn.Conv2d(cin, cout, 3, stride, 1, bias=False)


class PlainBlock(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.c1 = conv3(cin, cout, stride)
        self.b1 = nn.BatchNorm2d(cout)
        self.c2 = conv3(cout, cout)
        self.b2 = nn.BatchNorm2d(cout)
        self.short = None

    def forward(self, x):
        out = F.relu(self.b1(self.c1(x)))
        out = self.b2(self.c2(out))
        return F.relu(out)


class ResidualBlock(PlainBlock):
    def __init__(self, cin, cout, stride=1):
        super().__init__(cin, cout, stride)
        if stride != 1 or cin != cout:
            self.short = nn.Sequential(
                nn.Conv2d(cin, cout, 1, stride, bias=False),
                nn.BatchNorm2d(cout),
            )

    def forward(self, x):
        idt = x if self.short is None else self.short(x)
        out = F.relu(self.b1(self.c1(x)))
        out = self.b2(self.c2(out))
        return F.relu(out + idt)

두 블록의 `forward` 차이는 이게 전부입니다.

In [ ]:
import difflib
import inspect

diff = difflib.unified_diff(
    inspect.getsource(PlainBlock.forward).splitlines(),
    inspect.getsource(ResidualBlock.forward).splitlines(),
    "PlainBlock.forward", "ResidualBlock.forward",
    lineterm="", n=2)
print("\n".join(diff))

In [ ]:
class CifarNet(nn.Module):
    def __init__(self, n, block):
        super().__init__()
        self.stem = nn.Sequential(
            conv3(3, 16),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True))
        blocks, cin = [], 16
        for i, cout in enumerate((16, 32, 64)):
            for j in range(n):
                s = 2 if (i > 0 and j == 0) else 1
                blocks.append(block(cin, cout, s))
                cin = cout
        self.body = nn.Sequential(*blocks)
        self.head = nn.Linear(64, 10)

    def forward(self, x):
        x = self.body(self.stem(x))
        x = F.adaptive_avg_pool2d(x, 1).flatten(1)
        return self.head(x)


def build(depth, residual):
    block = ResidualBlock if residual else PlainBlock
    return CifarNet((depth - 2) // 6, block)


MODELS = [
    ("plain-20", 20, False),
    ("plain-56", 56, False),
    ("resnet-20", 20, True),
    ("resnet-56", 56, True),
]

for name, depth, res in MODELS:
    net = build(depth, res)
    n = sum(p.numel() for p in net.parameters())
    print(f"{name:<10} {n:>9,} 개")

## 학습 조건

네 모델에 **똑같이** 적용합니다. 한쪽에만 유리한 설정을 두면
깊이 비교가 아니라 설정 비교가 됩니다.

- SGD, 모멘텀 0.9, weight decay 1e-4, 배치 128
- 학습률 0.1 에서 시작해 코사인으로 0 까지 내림
- 앞 400 스텝은 선형 워밍업 — plain-56 이 초반에 발산하면
  degradation 과 구분이 안 되기 때문입니다. 네 모델 모두 켭니다.
- GPU 에서는 fp16 자동 혼합정밀도

In [ ]:
EPOCHS = int(os.environ.get("EPOCHS", 30))
BATCH = 128
LR = 0.1
WARMUP = 400
SEED = 0
ITERS = len(train_y) // BATCH

print("epochs      :", EPOCHS)
print("iters/epoch :", ITERS)
print("total iters :", EPOCHS * ITERS)


def err_rate(model, xs, ys, bs=1000):
    model.eval()
    wrong = torch.zeros((), device=dev)
    with torch.no_grad():
        for i in range(0, len(ys), bs):
            with torch.autocast(dev, enabled=amp):
                out = model(normalize(xs[i:i + bs]))
            wrong += (out.argmax(1) != ys[i:i + bs]).sum()
    return 100.0 * wrong.item() / len(ys)


def train(name, depth, residual):
    torch.manual_seed(SEED)
    model = build(depth, residual).to(dev)
    opt = torch.optim.SGD(model.parameters(), LR,
                          momentum=0.9,
                          weight_decay=1e-4)
    steps = EPOCHS * ITERS

    def factor(i):
        if i < WARMUP:
            return (i + 1) / WARMUP
        t = (i - WARMUP) / max(1, steps - WARMUP)
        return 0.5 * (1 + math.cos(math.pi * t))

    sch = torch.optim.lr_scheduler.LambdaLR(opt, factor)
    scaler = torch.amp.GradScaler(dev, enabled=amp)
    hist, t0 = [], time.time()
    for ep in range(EPOCHS):
        model.train()
        perm = torch.randperm(len(train_y), device=dev)
        wrong = torch.zeros((), device=dev)
        for i in range(ITERS):
            idx = perm[i * BATCH:(i + 1) * BATCH]
            x = augment(train_x[idx])
            y = train_y[idx]
            with torch.autocast(dev, enabled=amp):
                out = model(x)
                loss = F.cross_entropy(out, y)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            sch.step()
            wrong += (out.argmax(1) != y).sum()
        tr_err = 100.0 * wrong.item() / (ITERS * BATCH)
        te_err = err_rate(model, test_x, test_y)
        hist.append((ep + 1, tr_err, te_err))
        print(f"{name:<10} ep{ep + 1:>3}"
              f"  train {tr_err:5.2f}%"
              f"  test {te_err:5.2f}%")
    return {
        "name": name,
        "hist": hist,
        "sec": time.time() - t0,
        "iters": steps,
        "params": sum(p.numel()
                      for p in model.parameters()),
        "train_err": hist[-1][1],
        "test_err": hist[-1][2],
    }

## 학습 — 네 모델을 차례로

In [ ]:
runs = []
for name, depth, res in MODELS:
    runs.append(train(name, depth, res))
    print(f"{name:<10} 소요 {runs[-1]['sec'] / 60:.1f} 분\n")

## 결과 — 최종 오류율과 파라미터 수

In [ ]:
hdr = (f"{'model':<10}{'test err(%)':>12}"
       f"{'train err(%)':>13}{'params':>12}")
print(hdr)
print("-" * len(hdr))
for r in runs:
    print(f"{r['name']:<10}{r['test_err']:>12.2f}"
          f"{r['train_err']:>13.2f}{r['params']:>12,}")
print("-" * len(hdr))

err = {r["name"]: r["test_err"] for r in runs}
gap_p = err["plain-56"] - err["plain-20"]
gap_r = err["resnet-56"] - err["resnet-20"]
print(f"plain   20 -> 56 : {gap_p:+.2f} %p")
print(f"resnet  20 -> 56 : {gap_r:+.2f} %p")
print()
print("degradation :", "관찰됨" if gap_p > 0 else "관찰 안 됨")
print("역전        :", "관찰됨" if gap_r < 0 else "관찰 안 됨")

## 학습 곡선

가로는 학습 반복(iteration), 세로는 오류율(%)입니다.
옅은 선이 학습 오류, 진한 선이 테스트 오류입니다.

In [ ]:
import matplotlib.pyplot as plt

os.makedirs("out", exist_ok=True)
COLOR = {20: "tab:orange", 56: "tab:blue"}
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2),
                         sharey=True)
for ax, kind in zip(axes, ("plain", "resnet")):
    for r in runs:
        if not r["name"].startswith(kind):
            continue
        d = int(r["name"].split("-")[1])
        it = [e * ITERS for e, _, _ in r["hist"]]
        ax.plot(it, [h[1] for h in r["hist"]],
                color=COLOR[d], lw=1, alpha=0.35)
        ax.plot(it, [h[2] for h in r["hist"]],
                color=COLOR[d], lw=1.8,
                label=f"{kind}-{d}")
    ax.set_title(kind)
    ax.set_xlabel("iteration")
    ax.set_ylim(0, 60)
    ax.grid(alpha=0.25)
    ax.legend(loc="upper right")
axes[0].set_ylabel("error (%)")
fig.tight_layout()
fig.savefig("out/resnet_degradation_curves.png", dpi=150)
print("saved: out/resnet_degradation_curves.png")
plt.show()

## 읽는 법

- 위 표의 `plain 20 -> 56` 이 **양수**면 깊게 쌓아 더 나빠진
  것입니다. 층을 더한 만큼 표현력은 늘었는데 오류가 커졌다면
  최적화가 안 된 것이지 과적합이 아닙니다. 학습 오류를 같이
  보면 구분됩니다 — 과적합이면 학습 오류는 내려가야 합니다.
- `resnet 20 -> 56` 이 **음수**면 잔차 연결이 그 문제를
  뒤집은 것입니다.
- 이 노트북은 논문의 64k 반복이 아니라 위 셀에 적힌 반복
  수만큼만 돌립니다. 절대값은 논문과 다릅니다. 같은 조건에서
  깊이만 바꾼 **상대 비교**로 읽으세요.